# Advanced Company Brochure Maker

**LLM Engineering, Assignment 1.** A source-based information kit about two companies in the same field:
four Hebrew brochures (potential customers and job candidates for each company), a comparison table,
and a baseline brochure for the prompt experiment.

This notebook extends the class notebook `day5.ipynb` (collect links → choose with an LLM → collect content →
write a brochure → stream it). The logic lives in the `brochure/` package; this notebook runs it step by step,
shows the results, and holds the manual review.

![Architecture](docs/architecture.svg)

**How to run:** fill in the two companies in section 2, then run the cells in order.
A full run makes about 8 model calls: 2 link selections, 4 brochures, 1 comparison and 1 baseline.
Section 10 re-runs all checks from the saved files of a run, with no model calls.

## 1. Setup

In [1]:
from IPython.display import HTML, Markdown, display

from brochure import audit, config
from brochure.checks import check_saved_output, format_checks
from brochure.config import CONTEXT_CHAR_BUDGET, MODEL, Company, check_api_key, get_companies
from brochure.context import build_context, format_report
from brochure.generate import generate_brochure, generate_comparison, rtl_markdown
from brochure.links import build_candidates, save_candidates, select_links
from brochure.runlog import RunLog
from brochure.sources import (add_replacement_source, collect_sources, fetch_home, load_sources,
                              save_sources, summarize_sources)

check_api_key()
print(f"Model: {MODEL} | context budget: {CONTEXT_CHAR_BUDGET:,} characters of source text per call")

API key found and looks like a Claude API key.
Model: claude-opus-5-5 | context budget: 12,000 characters of source text per call


## 2. The two companies

Choose two companies **in the same field** whose sites can be read by the class scraper
(`scraper.py`). For each company we use the home page and up to 3 more pages, with no recursive crawling.

- `company_id`: a short slug used in file names (e.g. `"monday"`)
- `name`: the display name
- `url`: the home page

In [6]:
COMPANY_A = Company(company_id="ivory", name="אייבורי", url="https://www.ivory.co.il", letter="A")
COMPANY_B = Company(company_id="lastprice", name="לאסטפרייס", url="https://www.lastprice.co.il", letter="B")
FIELD = "קמעונאות מחשבים ואלקטרוניקה"

config.COMPANIES[:] = [COMPANY_A, COMPANY_B]
companies = get_companies()
assert all("TODO" not in c.company_id + c.name + c.url for c in companies), "Fill in both companies first."
for c in companies:
    print(f"{c.letter}: {c.name} ({c.url})  id={c.company_id}")
print("Field:", FIELD)

A: אייבורי (https://www.ivory.co.il)  id=ivory
B: לאסטפרייס (https://www.lastprice.co.il)  id=lastprice
Field: קמעונאות מחשבים ואלקטרוניקה


**Site choice notes (fill in).** Why these two companies, and any site that was tried and replaced:

| Company | Site | Readable by the scraper? | Notes (e.g. replaced because the site is JavaScript-only) |
|---|---|---|---|
| A | | | |
| B | | | |

## 3. Start a run

Everything from this run is saved under `runs/<run_id>/`: candidates, link selections, sources,
exact contexts and prompts, outputs with their metadata and checks, and the log (`log.jsonl`).

In [7]:
run = RunLog()
print("Run ID:", run.run_id)
print("Folder:", run.dir)

Run ID: 20261127-165418-3575
Folder: C:\Users\User\Documents\cursor\LLL_HM\LLM_HW1_Brochure_Maker\runs\20261127-165418-3575


## 4. Links and sources (per company)

**4a. Candidate links, built in code.** The home page is fetched once. Its links are resolved, `#fragments` are
removed, `mailto:` / `tel:` / `javascript:` and links to other sites are dropped (`www.` and the bare domain
count as the same site), the home page, privacy / terms pages and file links are dropped, and duplicates are
removed. The resulting list is saved, since it is exactly what the model receives.

In [8]:
homes, candidates, selections, collections = {}, {}, {}, {}

for co in companies:
    home = fetch_home(co, run=run)
    homes[co.company_id] = home
    print(f"{co.name}: HTTP {home['http_status']}, final URL {home['final_url']}, "
          f"{len(home['text']):,} chars of text, error: {home['error']}")
    assert home["error"] is None, f"The home page of {co.name} could not be read - choose another site."

    cands = build_candidates(co, home)
    save_candidates(cands, run.dir / "candidates")
    candidates[co.company_id] = cands
    print(f"  {cands['raw_link_count']} links on the page -> {len(cands['candidates'])} candidates "
          f"({len(cands['dropped'])} dropped, reasons saved)\n")

אייבורי: HTTP 200, final URL https://www.ivory.co.il/, 69,006 chars of text, error: None
  2399 links on the page -> 721 candidates (1678 dropped, reasons saved)

לאסטפרייס: HTTP 200, final URL https://www.lastprice.co.il/, 15,836 chars of text, error: None
  512 links on the page -> 487 candidates (25 dropped, reasons saved)



**4b. Link selection by the model (2 calls), validated in code.** The model picks up to 3 pages, each with a
`type`, `url` and `reason`. The answer is checked before anything is fetched. If anything is wrong, the whole
answer is rejected, its raw output and the reasons are saved, and nothing is fetched from it.
Re-running this cell retries only the rejected companies, and each retry is logged as a new attempt.

In [9]:
for co in companies:
    if selections.get(co.company_id, {}).get("status") == "accepted":
        continue
    sel = select_links(co, candidates[co.company_id], run=run)
    selections[co.company_id] = sel
    print(f"{co.name}: {sel['status']} (attempt {sel['attempt']}, {sel['duration_seconds']}s, tokens {sel['usage']})")
    for link in sel["links"]:
        print(f"  - [{link['type']}] {link['normalized_url']}\n      reason: {link['reason']}")
    for reason in sel["rejection_reasons"]:
        print("  ! rejected:", reason)
    print()

not_accepted = [co.name for co in companies if selections[co.company_id]["status"] != "accepted"]
if not_accepted:
    raise RuntimeError(f"Link selection was not accepted for {not_accepted}: nothing is fetched from it. "
                       "The raw output and reasons are in selections/. Re-run this cell to try again.")

אייבורי: accepted (attempt 1, 4.26s, tokens {'prompt_tokens': 22733, 'completion_tokens': 204, 'total_tokens': 22937})
  - [careers] https://www.ivory.co.il/jobs
      reason: Job openings and how to apply at Ivory, useful for candidates.
  - [stores] https://www.ivory.co.il/stores.html
      reason: Lists the physical store branches. Customers learn where to shop in person, and candidates see where the company operates.
  - [products] https://www.ivory.co.il/catalog.php?act=all_categories
      reason: A full overview of all product categories, from computers and phones to home appliances. Shows customers the whole range on offer.

לאסטפרייס: accepted (attempt 1, 3.0s, tokens {'prompt_tokens': 80280, 'completion_tokens': 173, 'total_tokens': 80453})
  - [about] https://www.lastprice.co.il/About
      reason: Company background and story of the online retailer - useful for both customers and candidates.
  - [careers] https://www.lastprice.co.il/Jobs
      reason: Open positions and how

**4c. Source collection.** The home page becomes A1 / B1 and the selected pages follow in the model's order
(A2–A4 / B2–B4). Each source is `ok`, `empty` or `failed`. Empty and failed sources stay in the record so
the gaps are visible, but they are never used as evidence.

In [10]:
for co in companies:
    col = collect_sources(co, homes[co.company_id], selections[co.company_id], run=run)
    save_sources(col, run.dir / "sources")
    collections[co.company_id] = col
    print(summarize_sources(col), "\n")

אייבורי (4 requests to the site)
  A1  ok       69006 chars  https://www.ivory.co.il/
  A2  ok       24298 chars  https://www.ivory.co.il/jobs
  A3  ok       20526 chars  https://www.ivory.co.il/stores.html
  A4  ok       27343 chars  https://www.ivory.co.il/catalog.php?act=all_categories
  usable sources: 4 of 4 

לאסטפרייס (4 requests to the site)
  B1  ok       15836 chars  https://www.lastprice.co.il/
  B2  ok       16543 chars  https://www.lastprice.co.il/About
  B3  ok       18283 chars  https://www.lastprice.co.il/Jobs
  B4  ok       19680 chars  https://www.lastprice.co.il/WhyLP
  usable sources: 4 of 4 



**If a company has too little usable information**, add a replacement page from its saved candidate list.
The replacement takes a free ID within the page limit, and the reason is recorded. Uncomment only if needed:

In [ ]:
# co = COMPANY_A
# add_replacement_source(collections[co.company_id], candidates[co.company_id],
#                        "https://...", "why this page is needed", run=run)
# save_sources(collections[co.company_id], run.dir / "sources")
# print(summarize_sources(collections[co.company_id]))

## 5. Contexts

All outputs are built from the **saved** sources, and nothing is fetched again. Only `ok` sources are used, within
a budget of source-text characters (`CONTEXT_CHAR_BUDGET` in `brochure/config.py`). The budget is split fairly,
so a long page is shortened instead of pushing the others out. Instructions, headers and source IDs are outside
the budget and are never cut. For the comparison, both companies share one budget.

In [11]:
saved = {co.company_id: load_sources(run.dir / "sources" / f"{co.company_id}.json") for co in companies}

contexts = {co.company_id: build_context([saved[co.company_id]]) for co in companies}
contexts["comparison"] = build_context([saved[COMPANY_A.company_id], saved[COMPANY_B.company_id]])

for name, ctx in contexts.items():
    print(f"[{name}]")
    print(format_report(ctx), "\n")

[ivory]
Source text used: 11,924 of 12,000 characters
  A1  shortened   2,981 / 69,006  shortened to its fair share of the budget (3,000 characters)
  A2  shortened   2,981 / 24,298  shortened to its fair share of the budget (3,000 characters)
  A3  shortened   2,981 / 20,526  shortened to its fair share of the budget (3,000 characters)
  A4  shortened   2,981 / 27,343  shortened to its fair share of the budget (3,000 characters) 

[lastprice]
Source text used: 11,996 of 12,000 characters
  B1  shortened   2,999 / 15,836  shortened to its fair share of the budget (3,000 characters)
  B2  shortened   2,999 / 16,543  shortened to its fair share of the budget (3,000 characters)
  B3  shortened   2,999 / 18,283  shortened to its fair share of the budget (3,000 characters)
  B4  shortened   2,999 / 19,680  shortened to its fair share of the budget (3,000 characters) 

[comparison]
Source text used: 11,960 of 12,000 characters
  A1  shortened   1,497 / 69,006  shortened to its fair share of 

## 6. Brochures (4 calls)

The same company facts serve two different readers. Each audience gets its own structure and emphasis.
Each brochure is checked in code right after it is generated: cited IDs, source list, word count (200–300 with `split`)
and required headings. The **first brochure is streamed**.

### 6.1 Company A, potential customers (streamed)

In [12]:
results = {}
results["A_customers"] = generate_brochure(COMPANY_A, "customers", contexts[COMPANY_A.company_id], run, stream=True)

<div dir="rtl">

# אייבורי: מידע ללקוחות פוטנציאליים

## הבעיה שהחברה פותרת
מי שמחפש מחשבים, סלולר או מוצרי חשמל צריך מקום לרכוש אותם ודרך לבדוק באיזה סניף קרוב יש מלאי. אייבורי מציגה את עצמה כ"הרשת המובילה למחשבים, סלולר ומוצרי חשמל" ומבטיחה "חוויה ייחודית של טכנולוגיה" [A1].

## מוצרים ושירותים
- **מחשבים, סלולר ומוצרי חשמל:** האתר מתאר את אייבורי כחנות מחשבים וסלולר [A1][A3] וכולל מפת אתר עם קטגוריות מוצרים [A4]. פירוט הקטגוריות עצמן: לא נמצא במקורות שנמסרו למודל.
- **בחירת סניף:** באתר אפשר לבחור סניף מתוך רשימה ארוכה, כדי להתאים את הקנייה לסניף הקרוב [A1][A3].
- **חיפוש לפי מלאי:** אפשר לסנן מלאי לפי סניף, עם האפשרויות "שווה", "גדול מ" ו"קטן מ". כך ניתן לבדוק זמינות לפני שמגיעים לחנות [A1][A4].
- **חנויות בפריסה ארצית:** יש סניפים, בין השאר, בתל אביב, ירושלים, חיפה, באר שבע, פתח תקווה, נתניה, אשדוד, כרמיאל, אילת וראש פינה [A3]. סניף אחד בהרצליה מסומן "לא פעיל" [A3].

## שימושים וקהלי יעד
- רכישת מחשבים, סלולר ומוצרי חשמל [A1].
- לקוחות שמעדיפים לקנות בחנות פיזית קרובה, מהצפון ועד אילת [A3].
- קהלי יעד מוגדרים (לקוחות פרטיים, עסקים או מוסדות): לא נמצא במקורות שנמסרו למודל.

## מה כדאי לברר לפני פנייה
- מה המחירים, והאם יש מבצעים או פריסת תשלומים? לא נמצא במקורות שנמסרו למודל.
- איזו אחריות ניתנת, והאם יש שירות תיקונים? לא נמצא במקורות שנמסרו למודל.
- האם יש משלוחים או איסוף עצמי, ומה זמני האספקה? לא נמצא במקורות שנמסרו למודל.
- האם יש מכירה לעסקים והצעות מחיר לכמויות? לא נמצא במקורות שנמסרו למודל.

## מקורות
- [A1] https://www.ivory.co.il/
- [A3] https://www.ivory.co.il/stores.html
- [A4] https://www.ivory.co.il/catalog.php?act=all_categories

</div>

[checks: ivory_customers] PASSED | words=242 (range 200-300) | cited=['A1', 'A3', 'A4'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x6


### 6.2 Company A, job candidates

In [13]:
results["A_candidates"] = generate_brochure(COMPANY_A, "candidates", contexts[COMPANY_A.company_id], run)

<div dir="rtl">

# אייבורי: מידע למועמדים לעבודה

## במה החברה עוסקת
אייבורי היא רשת קמעונאית בתחום המחשבים, הסלולר ומוצרי החשמל. לדברי החברה, היא "הרשת המובילה" בתחום [A1]. הפעילות נשענת על חנויות בפריסה ארצית [A3], ובאתר מופיעים מפת קטגוריות מוצרים [A4], בחירת סניף וחיפוש מלאי לפי סניף [A1].

## תפקידים וצוותים שפורסמו
לאתר יש עמוד "דרושים" [A2], אך בתוכן שנמסר ממנו לא הופיעו תפקידים, צוותים או תחומי עבודה מוגדרים: לא נמצא במקורות שנמסרו למודל.

## סביבת העבודה
- **תרבות, ערכים ודרך עבודה:** לא נמצא במקורות שנמסרו למודל.
- **מיקומים:** רשימת הסניפים באתר כוללת עשרות סניפים ברחבי הארץ [A1][A3], בהם:
  - תל אביב (השלושה, דיזינגוף סנטר, קרליבך, רמת אביב)
  - ירושלים (גבעת שאול, תלפיות, קינג ג'ורג')
  - חיפה, באר שבע, אשדוד, נתניה, כפר סבא ועפולה
  - כמה סניפים באילת
  - ראש פינה וכרמיאל [A3]
- **סניפים לוגיסטיים:** ברשימה מופיעים גם "פתח תקווה - סניף מחסן ראשי" ו"סניף מלאי חיצוני" [A1]. סניף בהרצליה מסומן ברשימה כ"לא פעיל" [A1].
- **עבודה מרחוק:** לא נמצא במקורות שנמסרו למודל.

## תהליך המועמדות
- **איך מגישים:** לחברה יש עמוד משרות בכתובת https://www.ivory.co.il/jobs [A2]. אופן הגשת המועמדות ושלבי התהליך: לא נמצא במקורות שנמסרו למודל.
- **שאלות שכדאי לשאול בראיון:**
  - באילו סניפים או מחסנים מתבצעת העבודה בתפקיד, ומה מבנה המשמרות?
  - איזו הכשרה מקצועית על מוצרי מחשבים וסלולר ניתנת לעובדים חדשים?

## מקורות
- [A1] https://www.ivory.co.il/
- [A2] https://www.ivory.co.il/jobs
- [A3] https://www.ivory.co.il/stores.html
- [A4] https://www.ivory.co.il/catalog.php?act=all_categories

</div>

[checks: ivory_candidates] PASSED | words=211 (range 200-300) | cited=['A1', 'A2', 'A3', 'A4'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x4


### 6.3 Company B, potential customers

In [14]:
results["B_customers"] = generate_brochure(COMPANY_B, "customers", contexts[COMPANY_B.company_id], run)

<div dir="rtl">

# לאסטפרייס: מידע ללקוחות פוטנציאליים

## הבעיה שהחברה פותרת
הקורא מחפש מקום אחד לקנות בו מוצרי חשמל, סלולר ומוצרים לבית, במחיר נמוך. החברה מציגה את עצמה כאתר "קניות באינטרנט במחירים ללא תחרות" [B1][B4].

## מוצרים ושירותים
- **קטלוג מקוון רחב:** אפשר לרכוש מוצרים במחלקות כמו מוצרי חשמל קטנים למטבח, מקררים ומקפיאים, מכונות ומייבשי כביסה, מיזוג ואקלים, טלוויזיות, סלולר, מחשוב וגיימינג, אוזניות וסאונד, עולם הקפה וטיפוח ויופי [B1].
- **מבצעים:** באתר יש מדור "דילים לוהטים" וקישור לערוץ טלגרם [B1].
- **אזור אישי:** אפשר להירשם ולהתחבר לחשבון אישי, גם בהתחברות מהירה [B1].
- **סניפים פיזיים:** לדברי החברה, "החוויה מהאינטרנט עכשיו בחנויות" [B1].
- **שירות ותמיכה:** יש מרכז שירות, נציג אנושי ב-WhatsApp ומוקד מכירות ושירות בטלפון 1-700-70-80-89 [B1].

## שימושים וקהלי יעד
- **משקי בית:** ציוד למטבח, לכביסה, לניקיון, לאפייה ובישול ולמיזוג [B1].
- **פנאי ותחביבים:** מחלקות ספורט, לגן, חיות מחמד ועולם הגיימינג [B1].
- **עסקים:** באתר יש אזור "לאסטפרייס לעסקים" [B1]. פרטי השירות לעסקים: לא נמצא במקורות שנמסרו למודל.

## מה כדאי לברר לפני פנייה
- **מחירים:** מהם המחירים ותנאי התשלום בפועל, ואיך הם משתווים למתחרים? לא נמצא במקורות שנמסרו למודל.
- **משלוח והתקנה:** מהן העלויות וזמני האספקה, ובמיוחד למוצרים גדולים כמו מקררים ומזגנים? לא נמצא במקורות שנמסרו למודל.
- **אחריות והחזרות:** מהי מדיניות האחריות, ההחזרות וההחלפות? לא נמצא במקורות שנמסרו למודל.
- **סניפים:** היכן הסניפים, מהן שעות הפעילות, והאם המחירים בהם זהים למחירי האתר? לא נמצא במקורות שנמסרו למודל.

## מקורות
- [B1] https://www.lastprice.co.il/
- [B4] https://www.lastprice.co.il/WhyLP

</div>

[checks: lastprice_customers] PASSED | words=228 (range 200-300) | cited=['B1', 'B4'] of included=['B1', 'B2', 'B3', 'B4'] | missing-info phrase x5


### 6.4 Company B, job candidates

In [15]:
results["B_candidates"] = generate_brochure(COMPANY_B, "candidates", contexts[COMPANY_B.company_id], run)

<div dir="rtl">

# לאסטפרייס: מידע למועמדים לעבודה

## במה החברה עוסקת
לאסטפרייס מפעילה אתר קניות באינטרנט. באתר יש מחלקות רבות, ובהן מוצרי חשמל קטנים למטבח, מיזוג ואקלים, מקררים, סלולר ומחשוב, אוזניות וסאונד, עולם הקפה וטיפוח ויופי [B1]. לצד האתר מוזכרים סניפים פיזיים ("החוויה מהאינטרנט עכשיו בחנויות"), מוקד מכירות ושירות ופעילות בשם "לאסטפרייס לעסקים" [B1]. מכאן שהעבודה בחברה נוגעת למסחר מקוון, לחנויות ולשירות לקוחות [B1].

## תפקידים וצוותים שפורסמו
- **משרות פתוחות:** לא נמצא במקורות שנמסרו למודל. לחברה יש עמוד דרושים שכותרתו "דרושים אנשים שרוצים להיות חלק מהעתיד | לאסטפרייס מחפשת אותך", אך תוכנו לא נכלל במקורות [B3].
- **תחומי פעילות שמוזכרים באתר** (לא כמשרות):
  - מוקד המכירות והשירות ("צוות Total") [B1]
  - מרכז השירות והתמיכה [B1]
  - נציגים אנושיים ב-WhatsApp [B1]
  - הסניפים [B1]
  - לאסטפרייס לעסקים [B1]

## סביבת העבודה
לא נמצא במקורות שנמסרו למודל. באתר יש עמוד בשם "הצוות מאחורי החוויה", שלדברי החברה מציג את "האנשים מאחורי לאסטפרייס", אך תוכנו לא נכלל במקורות [B2]. גם מיקומי הסניפים ואפשרויות עבודה מרחוק: לא נמצא במקורות שנמסרו למודל.

## תהליך המועמדות
לא נמצא במקורות שנמסרו למודל. מומלץ לבדוק ישירות את עמוד הדרושים באתר [B3].

שאלות שכדאי לשאול בראיון:
- לאילו צוותים מגייסים כעת: האתר, מוקד המכירות והשירות, הסניפים או לאסטפרייס לעסקים?
- איך משתלבת העבודה של הצוותים המקוונים עם הסניפים ועם מוקד השירות?

## מקורות
- [B1] https://www.lastprice.co.il/
- [B2] https://www.lastprice.co.il/About
- [B3] https://www.lastprice.co.il/Jobs

</div>

[checks: lastprice_candidates] PASSED | words=210 (range 200-300) | cited=['B1', 'B2', 'B3'] of included=['B1', 'B2', 'B3', 'B4'] | missing-info phrase x4


## 7. Comparison (1 call)

Four dimensions fixed before generation (`COMPARISON_DIMENSIONS` in `brochure/config.py`): products and services,
target audiences, pricing or trial, and working at the company. It is based on the companies' own publications, so it
is not independent market research.

In [16]:
results["comparison"] = generate_comparison(COMPANY_A, COMPANY_B, contexts["comparison"], run)

<div dir="rtl">

# השוואה: אייבורי מול לאסטפרייס

| ממד | אייבורי | לאסטפרייס |
|---|---|---|
| מוצרים ושירותים | החברה מציגה את עצמה כ"רשת המובילה למחשבים, סלולר ומוצרי חשמל" [A1] וכ"חנות מחשבים וסלולר" [A3][A4]. באתר מופיעה רשימת סניפים רבים ברחבי הארץ (סניף אחד מסומן "לא פעיל") וכלי "חיפוש לפי מלאי" לפי סניף [A1][A3]. | תפריט מחלקות רחב: מוצרי חשמל למטבח, מיזוג, מקררים, כביסה, סלולר ומחשוב, סאונד, טלוויזיות ומקרנים, קפה, טיפוח, ספורט, בית, גן וחיות מחמד [B1][B4]. שירות דרך מרכז שירות, נציג ב-WhatsApp, מוקד טלפוני 1-700-70-80-89 וסניפים ("החוויה מהאינטרנט עכשיו בחנויות") [B1]. |
| קהלי יעד | לא נמצא במקורות שנמסרו למודל | בניווט מופיע קישור "לאסטפרייס לעסקים" לצד הרשמה לאזור אישי [B1][B2]. אין פירוט נוסף על הקהלים. |
| תמחור או תקופת ניסיון | לא נמצא במקורות שנמסרו למודל | לא מופיעים מחירים או תנאים. החברה מתארת "מחירים ללא תחרות" [B1][B4], ובאתר יש מדור "דילים לוהטים" [B1]. |
| עבודה בחברה | קיים עמוד בשם "דרושים" [A2], אך בתוכן שנמסר אין משרות או תנאי העסקה. | קיים עמוד גיוס: "דרושים אנשים שרוצים להיות חלק מהעתיד" [B3], ועמוד "הצוות מאחורי החוויה" [B2]. בתוכן שנמסר אין משרות או תנאים. |

## הבדלים עיקריים
- **היקף המוצרים:** לאסטפרייס מציגה תפריט מחלקות מפורט שכולל גם תחומים כמו גן, ספורט וחיות מחמד [B1]. אצל אייבורי יש רק תיאור כללי של מחשבים, סלולר ומוצרי חשמל [A1]. עמוד הקטגוריות של אייבורי נמסר בקיצור, ולא הופיעה בו רשימת קטגוריות [A4]. לכן ייתכן שהפער נובע מהחומר שנמסר ולא מהמוצרים עצמם.
- **ערוצי מכירה ושירות:** אצל אייבורי בולטת רשת סניפים רשומה עם חיפוש מלאי לפי סניף [A1][A3]. לאסטפרייס מדגישה ערוצים מקוונים וטלפוניים, ומזכירה סניפים ללא רשימה [B1].
- **תמחור:** אי אפשר להשוות. אצל אייבורי אין מידע, ואצל לאסטפרייס יש רק טענה שיווקית ללא מחירים או תנאים [B1][B4].
- **עבודה בחברה:** לשתי החברות יש עמודי גיוס [A2][B3], אך אין בהם פרטים שמאפשרים השוואה.

## למי כל חברה עשויה להתאים
אייבורי עשויה להתאים למי שמחפש מחשבים וסלולר ורוצה לבדוק מלאי בסניף פיזי קרוב [A1][A3]. לאסטפרייס עשויה להתאים למי שמחפש מגוון רחב של מוצרי חשמל ובית ושירות דרך WhatsApp או טלפון, וכן לעסקים [B1]. לפני החלטה כדאי לבדוק מחירים בפועל, תנאי משלוח, אחריות והחזרות, את רשימת הסניפים של לאסטפרייס ואת המשרות הפתוחות בשתי החברות.

ההשוואה מבוססת על פרסומי החברות עצמן בלבד, ולא על מחקר שוק בלתי תלוי. העובדה שבאתר של חברה אחת מופיעים יותר פרטים אינה מעידה שהיא טובה יותר.

## מקורות
- A1: https://www.ivory.co.il/
- A2: https://www.ivory.co.il/jobs
- A3: https://www.ivory.co.il/stores.html
- A4: https://www.ivory.co.il/catalog.php?act=all_categories
- B1: https://www.lastprice.co.il/
- B2: https://www.lastprice.co.il/About
- B3: https://www.lastprice.co.il/Jobs
- B4: https://www.lastprice.co.il/WhyLP

</div>

[checks: comparison] FAILED | cited=['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4'] of included=['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4'] | missing-info phrase x2
  - cited but missing from the source list: ['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4']


## 8. Baseline brochure for the prompt experiment (1 call)

Same company, audience, model, settings and **exact same context** as brochure 6.1, with the same language, length and
source / citation rules. Only the prompt wording and the requested structure change (the class notebook's simple prompt).

In [ ]:
results["A_customers_baseline"] = generate_brochure(
    COMPANY_A, "customers", contexts[COMPANY_A.company_id], run, variant="baseline")

## 9. Run summary

In [ ]:
rows = ["| Output | Run ID | Status | Words | Checks |", "|---|---|---|---|---|"]
for r in results.values():
    words = r["word_count"] if r["word_count"] is not None else "-"
    checks = "passed" if r["checks"]["passed"] else "; ".join(r["checks"]["problems"])
    rows.append(f"| {r['name']} | {r['run_id']} | {r['status']} | {words} | {checks} |")
display(Markdown("\n".join(rows)))

print(run.format_summary())
run.save_summary()

If a brochure fails a check (for example, the word count), keep the original output and either document it or
regenerate it. A regeneration is logged as attempt 2, saved as `<name>_attempt2`, and overwrites nothing.

## 10. Replay from saved files (no model calls)

Re-runs the checks on every saved output, and rebuilds every context from the saved sources to show it matches the
context that was actually sent. Set `REPLAY_RUN_ID` to any earlier run ID to re-check that run without calling the model.

In [ ]:
REPLAY_RUN_ID = run.run_id          # or an earlier run, e.g. "20261127-120000-abcd"
replay = RunLog(REPLAY_RUN_ID)

import json
saved_sources = {p.stem: load_sources(p) for p in (replay.dir / "sources").glob("*.json")}
for meta_path in sorted((replay.dir / "outputs").glob("*.json")):
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    name = meta_path.stem
    print(f"{name}: {format_checks(check_saved_output(replay.dir, name))}")

    ids = meta["company_id"].split("+")              # "a" for a brochure, "a+b" for the comparison
    rebuilt = build_context([saved_sources[i] for i in ids])
    sent = (replay.dir / "contexts" / f"{name}.txt").read_text(encoding="utf-8")
    print(f"   context rebuilt from saved sources == context sent: {rebuilt['text'] == sent}")

## 11. Manual claim audit (8 claims)

**Sampling rule (fixed in advance):** in each of the 4 improved brochures, take the **first two factual claims** in
reading order. A factual claim is a sentence or bullet in the body with at least one citation that is not a
"לא נמצא במקורות שנמסרו למודל" statement. Headings, the source list and table rows are not claims.
If a brochure has fewer than two, that is recorded as a gap.

The cell below applies the rule to the saved brochures and shows lines from the cited sources that share words with each
claim. That is only a reading aid: the verdict comes from reading the source. A cited ID existing is **not** proof that the
source supports the claim.

In [ ]:
AUDIT_NAMES = [f"{co.company_id}_{aud}" for co in companies for aud in ("customers", "candidates")]
claims = audit.sample_claims(replay.dir, AUDIT_NAMES, per_brochure=2)
print(audit.format_claims(claims))

audit_path = replay.dir / "audit" / "claim_audit.csv"
if not audit_path.exists():
    audit.write_audit_template(claims, audit_path)
print("Fill in evidence / verdict / note in:", audit_path)
print("Verdicts:", " | ".join(audit.VERDICTS))

**Audit results (fill in):**

| # | Brochure | Claim (as written) | Source ID | URL | Evidence quote from the source | Verdict | Correction or explanation |
|---|---|---|---|---|---|---|---|
| 1 | | | | | | | |
| 2 | | | | | | | |
| 3 | | | | | | | |
| 4 | | | | | | | |
| 5 | | | | | | | |
| 6 | | | | | | | |
| 7 | | | | | | | |
| 8 | | | | | | | |

The original outputs stay unchanged in `outputs/`. Any correction is written here.

## 12. Prompt experiment: baseline vs improved

**Criteria, fixed before reading the outputs:**
1. **Faithfulness:** every claim is supported by the cited source, with nothing invented and marketing claims attributed.
2. **Audience fit:** structure, order and explanations serve a potential customer.
3. **Information coverage:** problem, offering, uses / audiences, and open questions.
4. **Clarity:** easy to scan, with clear headings and no filler.

The same model, settings, context, audience, language, length and source rules were used for both. Only the
prompt wording and the requested structure differ. This is one small comparison, not a general ranking of prompts or models.

In [ ]:
improved, baseline = results["A_customers"], results["A_customers_baseline"]
for label, r in (("Improved", improved), ("Baseline", baseline)):
    print(f"{label}: {r['word_count']} words, cited {r['checks']['cited_ids']}, "
          f"'not found' x{r['checks']['missing_info_mentions']}, checks {'passed' if r['checks']['passed'] else r['checks']['problems']}")

def column(title, text):
    return (f'<div style="flex:1; min-width:300px; border:1px solid #ccc; border-radius:8px; padding:8px 16px">'
            f'<h3 dir="ltr">{title}</h3>\n\n{rtl_markdown(text).data}\n\n</div>')

display(Markdown('<div style="display:flex; gap:16px; flex-wrap:wrap">\n\n'
                 + column("Improved", improved["text"]) + column("Baseline", baseline["text"]) + "\n\n</div>"))

**Findings (fill in, with concrete examples quoted from both versions):**

| Criterion | Improved | Baseline | Example |
|---|---|---|---|
| Faithfulness | | | |
| Audience fit | | | |
| Information coverage | | | |
| Clarity | | | |

Where the improved prompt did **not** help, say so too.

## 13. Discussion (to complete)

- **Code vs model:** what the code does (links, validation, fetching, budget, checks, logging) and what the model does
  (choosing pages, writing).
- **Context management:** the character budget, fair split, what was shortened or left out in this run.
- **Validation points:** link selection, source status, citations / source list / word count, and the manual audit.
- **Changing the prompt vs changing the source material:** how each can change the output.
- **Workflow vs agent:** this is a fixed workflow. What would be needed for an agent that chooses tools at run time.